# Week 5 - PySpark Data Cleaning and Transformations


## 1. Introduction

## 2. Objective

The objective of this assignment is to understand the fundamentals of Apache Spark and perform data cleaning, transformation, and aggregation using Spark DataFrames. The assignment also demonstrates filtering, grouping, schema modification, and building a simple data processing pipeline.

## 3. Import libraries

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## 4. Create Park Section
SparkSession is the entry point for working with Spark DataFrames

In [3]:
spark = SparkSession.builder \
    .appName("Week5_PySpark_Assignment") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


In [4]:
spark

## 5. Load Dataset

The Sample Superstore dataset is loaded into a Spark DataFrame for further analysis and transformations.

In [5]:
test_df = spark.createDataFrame(
    [
        (1, "Alice"),
        (2, "Bob"),
        (3, "Charlie")
    ],
    ["ID", "Name"]
)

test_df.show()

+---+-------+
| ID|   Name|
+---+-------+
|  1|  Alice|
|  2|    Bob|
|  3|Charlie|
+---+-------+



In [8]:
df = spark.read.csv(
    "../data/SampleSuperstore.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

### Load Additional Datasets

The datasets are loaded into Spark DataFrames for performing data cleaning, transformation, and aggregation tasks.

In [19]:
orders_df = spark.read.csv(
    "../data/orders_dirty.csv",
    header=True,
    inferSchema=True
)

customers_df = spark.read.csv(
    "../data/customers_dirty.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "../data/products_dirty.csv",
    header=True,
    inferSchema=True
)

order_items_df = spark.read.csv(
    "../data/order_items_dirty.csv",
    header=True,
    inferSchema=True
)

# 6. Explore Dataset

In [9]:
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [10]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [11]:
print(f"Total Rows: {df.count()}")

Total Rows: 9994


In [12]:
print(f"Total Columns: {len(df.columns)}")

Total Columns: 21


In [13]:
print(df.columns)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [14]:
df.describe().show()

+-------+------------------+--------------+----------+---------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|summary|            Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|    City|  State|       Postal Code| Region|     Product ID|  Category|Sub-Category|        Product Name|             Sales|          Quantity|          Discount|            Profit|
+-------+------------------+--------------+----------+---------+--------------+-----------+------------------+-----------+-------------+--------+-------+------------------+-------+---------------+----------+------------+--------------------+------------------+------------------+------------------+------------------+
|  count|              9994|          9994|   

#### Orders Dataset

In [20]:
orders_df.show(5)

orders_df.printSchema()
print(f"Orders Records : {orders_df.count()}")

+---------+-----------+-------------------+---------+-----------+
| order_id|customer_id|         order_date|   status|region_code|
+---------+-----------+-------------------+---------+-----------+
|ORD000001|  CUST00961|2025-05-19 12:50:57|   PLACED|       EAST|
|ORD000002|  CUST00573|2023-07-13 10:52:55|CANCELLED|       EAST|
|ORD000003|  CUST00833|2024-06-07 00:41:07|  SHIPPED|    CENTRAL|
|ORD000004|  CUST00096|2023-07-29 15:18:24|CANCELLED|    CENTRAL|
|ORD000005|  CUST00371|2024-07-09 00:00:00|   PLACED|       EAST|
+---------+-----------+-------------------+---------+-----------+
only showing top 5 rows
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- region_code: string (nullable = true)

Orders Records : 5000


#### Customers Dataset

In [21]:
customers_df.show(5)

+-----------+-------------+--------------------+-----------------+-------------+
|customer_id|customer_name|               email|registration_date|customer_type|
+-----------+-------------+--------------------+-----------------+-------------+
|  CUST00001|Aryan Maharaj|udantdewan@exampl...|       2023-12-03|          VIP|
|  CUST00002|  Pahal Balay|   invalid_email.com|       2025-08-20|      REGULAR|
|  CUST00003| Rushil Saini|saumyamall@exampl...|       2024-03-01|      REGULAR|
|  CUST00004|  Harini Mall|tanveernayar@exam...|       2023-11-18|          VIP|
|  CUST00005|Gunbir Parmer|aishani07@example...|       2024-01-06|      PREMIUM|
+-----------+-------------+--------------------+-----------------+-------------+
only showing top 5 rows


In [22]:
customers_df.printSchema()


root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- customer_type: string (nullable = true)



In [23]:
print(f"Customers Records : {customers_df.count()}")

Customers Records : 1010


#### Products Dataset

In [24]:
products_df.show(5)

+----------+---------------+--------+-----------+----------+
|product_id|   product_name|category|subcategory|cost_price|
+----------+---------------+--------+-----------+----------+
| PROD00001|Philips Outdoor|  Sports|    Outdoor|     39267|
| PROD00002|   Sony Fitness|  Sports|    Fitness|     34447|
| PROD00003|    LG Academic|   Books|   Academic|     29214|
| PROD00004|     Puma Decor|    Home|      Decor|     20213|
| PROD00005| Samsung Indoor|  Sports|     Indoor|     40152|
+----------+---------------+--------+-----------+----------+
only showing top 5 rows


In [25]:
products_df.printSchema()


root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- cost_price: integer (nullable = true)



In [26]:
print(f"Products Records : {products_df.count()}")


Products Records : 250


#### order items dataset

In [27]:

order_items_df.show(5)

+----------+---------+----------+--------+----------+----------------+
|   item_id| order_id|product_id|quantity|unit_price|discount_percent|
+----------+---------+----------+--------+----------+----------------+
|ITEM000001|ORD002900| PROD00135|       1|     13320|              25|
|ITEM000002|ORD004612| PROD00221|       5|     51413|               5|
|ITEM000003|ORD004368| PROD00093|       5|     34709|               0|
|ITEM000004|ORD004349| PROD00043|       5|     15401|              25|
|ITEM000005|ORD001357| PROD00163|       3|     10703|               0|
+----------+---------+----------+--------+----------+----------------+
only showing top 5 rows


In [28]:
order_items_df.printSchema()


root
 |-- item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)



In [29]:
print(f"Order Items Records : {order_items_df.count()}")


Order Items Records : 15000


### Observation

The dataset contains order details, customer information, product details, sales, quantity, discount, and profit. The data has been loaded successfully into a Spark DataFrame and is ready for further cleaning and transformation.

All four datasets were loaded successfully into Spark DataFrames. The datasets represent customers, orders, products, and order items, forming a simple e-commerce data model that can be used for data cleaning and analytical operations.

# 7. Spark Fundamentals (Q1 & Q2)

### Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Traditional MapReduce writes intermediate results to disk after every processing stage, making it slower for iterative tasks. It also requires multiple jobs for complex workflows, which increases execution time. Apache Spark overcomes these limitations by performing most operations in memory, resulting in faster processing and better performance for modern big data applications.

### Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems. 

Spark stores intermediate data in memory instead of writing it to disk after every step. This significantly reduces disk I/O and improves the performance of iterative algorithms such as machine learning and graph processing, where the same data is processed multiple times.

#  8. Data Cleaning (Q3, Q4, Q5, Q7, Q10, Q12, Q14)

### Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date. 

In [31]:
removed_duplicates = df.dropDuplicates(["Order ID", "Order Date"])

print("Original Rows :", df.count())
print("Rows After Removing Duplicates :", removed_duplicates.count())

Original Rows : 9994
Rows After Removing Duplicates : 5009


In [32]:
removed_duplicates.show(5)

+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+------------+--------+--------+--------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|   Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|       Sales|Quantity|Discount|  Profit|
+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+----------+-----------+------+---------------+---------------+------------+--------------------+------------+--------+--------+--------+
|  2718|CA-2014-100006|  9/7/2014|9/13/2014|Standard Class|   DK-13375|     Dennis Kane|   Consumer|United States|New York City|  New York|      10024|  East|TEC-PH-10002075|     Technology|      Phones|  

### Observation

Duplicate records were removed based on Order ID and Order Date. This helps maintain data consistency and prevents repeated records from affecting business analysis.

## Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'. 

The .na.drop() function removes rows containing null values, whereas .na.fill() replaces null values with a specified value instead of removing the row.

In [36]:
customers_df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in customers_df.columns]
).show()

+-----------+-------------+-----+-----------------+-------------+
|customer_id|customer_name|email|registration_date|customer_type|
+-----------+-------------+-----+-----------------+-------------+
|          0|            0|    0|                0|            0|
+-----------+-------------+-----+-----------------+-------------+



In [37]:
demo_data = [
    ("C001", "Active"),
    ("C002", None),
    ("C003", "Inactive"),
    ("C004", None)
]

demo_df = spark.createDataFrame(
    demo_data,
    ["customer_id", "status"]
)

demo_df.show()

+-----------+--------+
|customer_id|  status|
+-----------+--------+
|       C001|  Active|
|       C002|    NULL|
|       C003|Inactive|
|       C004|    NULL|
+-----------+--------+



In [38]:
demo_df.na.drop().show()

+-----------+--------+
|customer_id|  status|
+-----------+--------+
|       C001|  Active|
|       C003|Inactive|
+-----------+--------+



In [39]:
demo_df.na.fill({"status": "Unknown"}).show()

+-----------+--------+
|customer_id|  status|
+-----------+--------+
|       C001|  Active|
|       C002| Unknown|
|       C003|Inactive|
|       C004| Unknown|
+-----------+--------+



## Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them? 

Spark DataFrames are immutable, meaning they cannot be modified directly. Every transformation such as filtering, renaming columns, or dropping columns creates a new DataFrame while leaving the original DataFrame unchanged.

## Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()? 

Handling null values before performing aggregation helps improve the accuracy of calculations. Missing values may lead to incorrect results or incomplete analysis if they are not managed properly.

## Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time. 

In [43]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- region_code: string (nullable = true)



In [44]:
from pyspark.sql.functions import to_timestamp

orders_timestamp = (
    orders_df
    .withColumn(
        "event_time",
        to_timestamp(col("order_date"), "dd-MM-yyyy HH:mm")
    )
    .drop("order_date")
)

orders_timestamp.show(5)

+---------+-----------+---------+-----------+-------------------+
| order_id|customer_id|   status|region_code|         event_time|
+---------+-----------+---------+-----------+-------------------+
|ORD000001|  CUST00961|   PLACED|       EAST|2025-05-19 12:50:57|
|ORD000002|  CUST00573|CANCELLED|       EAST|2023-07-13 10:52:55|
|ORD000003|  CUST00833|  SHIPPED|    CENTRAL|2024-06-07 00:41:07|
|ORD000004|  CUST00096|CANCELLED|    CENTRAL|2023-07-29 15:18:24|
|ORD000005|  CUST00371|   PLACED|       EAST|2024-07-09 00:00:00|
+---------+-----------+---------+-----------+-------------------+
only showing top 5 rows


In [45]:
orders_timestamp.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- region_code: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



### Observation

The order_date column was converted into TimestampType and renamed to event_time, making it suitable for time-based analysis.

## Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string. 

In [46]:
customers_df.show(5)

+-----------+-------------+--------------------+-----------------+-------------+
|customer_id|customer_name|               email|registration_date|customer_type|
+-----------+-------------+--------------------+-----------------+-------------+
|  CUST00001|Aryan Maharaj|udantdewan@exampl...|       2023-12-03|          VIP|
|  CUST00002|  Pahal Balay|   invalid_email.com|       2025-08-20|      REGULAR|
|  CUST00003| Rushil Saini|saumyamall@exampl...|       2024-03-01|      REGULAR|
|  CUST00004|  Harini Mall|tanveernayar@exam...|       2023-11-18|          VIP|
|  CUST00005|Gunbir Parmer|aishani07@example...|       2024-01-06|      PREMIUM|
+-----------+-------------+--------------------+-----------------+-------------+
only showing top 5 rows


In [47]:
customers_clean = customers_df.filter(
    col("email").isNotNull() &
    (trim(col("customer_name")) != "")
)

customers_clean.show(5)

+-----------+-------------+--------------------+-----------------+-------------+
|customer_id|customer_name|               email|registration_date|customer_type|
+-----------+-------------+--------------------+-----------------+-------------+
|  CUST00001|Aryan Maharaj|udantdewan@exampl...|       2023-12-03|          VIP|
|  CUST00002|  Pahal Balay|   invalid_email.com|       2025-08-20|      REGULAR|
|  CUST00003| Rushil Saini|saumyamall@exampl...|       2024-03-01|      REGULAR|
|  CUST00004|  Harini Mall|tanveernayar@exam...|       2023-11-18|          VIP|
|  CUST00005|Gunbir Parmer|aishani07@example...|       2024-01-06|      PREMIUM|
+-----------+-------------+--------------------+-----------------+-------------+
only showing top 5 rows


### Observation

Rows with missing email addresses or empty customer names were removed to improve data quality before further analysis.

## Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

If the dataset contains inconsistent or incorrect date formats, Spark may assign the wrong data type to a column. This can lead to errors during filtering, aggregation, or timestamp conversion. It is often better to define the schema explicitly for production datasets.

# 9. Data Transformations & Aggregation ( Q4, Q6, Q8, Q11, Q13 )

## Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'. 

In [41]:
subscription_data = [
    ("Amit", 22, "Premium"),
    ("Riya", 19, "Basic"),
    ("Karan", 28, "Premium"),
    ("Neha", 35, "Premium"),
    ("Rahul", 25, "Premium")
]

subscription_df = spark.createDataFrame(
    subscription_data,
    ["name", "age", "subscription"]
)

subscription_df.show()

+-----+---+------------+
| name|age|subscription|
+-----+---+------------+
| Amit| 22|     Premium|
| Riya| 19|       Basic|
|Karan| 28|     Premium|
| Neha| 35|     Premium|
|Rahul| 25|     Premium|
+-----+---+------------+



In [42]:
premium_users = subscription_df.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)

premium_users.show()

+-----+---+------------+
| name|age|subscription|
+-----+---+------------+
| Amit| 22|     Premium|
|Karan| 28|     Premium|
|Rahul| 25|     Premium|
+-----+---+------------+



### Observation

The filter() function was used with multiple conditions to retrieve only Premium users whose age lies between 18 and 30 years.

### Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount. 

In [ ]:
sales_df = (
    orders_df
    .join(order_items_df, "order_id")
    .join(products_df, "product_id")
)

In [ ]:
sales_df.show(5)

+----------+---------+-----------+-------------------+---------+-----------+----------+--------+----------+----------------+-----------------+-----------+-----------+----------+
|product_id| order_id|customer_id|         order_date|   status|region_code|   item_id|quantity|unit_price|discount_percent|     product_name|   category|subcategory|cost_price|
+----------+---------+-----------+-------------------+---------+-----------+----------+--------+----------+----------------+-----------------+-----------+-----------+----------+
| PROD00135|ORD002900|  CUST00146|2023-09-17 07:07:49|DELIVERED|       WEST|ITEM000001|       1|     13320|              25|   HP Accessories|Electronics|Accessories|     42254|
| PROD00221|ORD004612|  CUST00382|2024-06-03 00:46:23|CANCELLED|       EAST|ITEM000002|       5|     51413|               5|       Apple Kids|   Clothing|       Kids|     11809|
| PROD00093|ORD004368|  CUST00037|2023-10-21 11:29:12|   PLACED|    CENTRAL|ITEM000003|       5|     34709|   

In [ ]:
west_sales = (
    sales_df
    .filter(col("region_code") == "WEST")
    .groupBy("category")
    .agg(avg("unit_price").alias("Average_Sale_Amount"))
)

west_sales.show()

+-----------+-------------------+
|   category|Average_Sale_Amount|
+-----------+-------------------+
|       Home| 28732.039076376554|
|     Sports| 30045.824561403508|
|Electronics| 29294.268018018018|
|   Clothing|  30536.88411214953|
|      Books| 29922.410029498526|
|     Beauty| 30840.022304832713|
+-----------+-------------------+



### Observation

The three datasets were joined to create a consolidated sales dataset. Records belonging to the WEST region were filtered, and the average sale amount was calculated for each product category.

## Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100. 

In [ ]:
region_count = (
    df
    .groupBy("city")
    .count()
    .filter(col("count") > 100)
)

region_count.show()

+-------------+-----+
|         city|count|
+-------------+-----+
|  Springfield|  163|
|       Dallas|  157|
| Philadelphia|  537|
|  Los Angeles|  747|
|San Francisco|  510|
|    San Diego|  170|
|      Detroit|  115|
|     Columbus|  222|
|      Chicago|  314|
|      Seattle|  428|
|New York City|  915|
|      Houston|  377|
| Jacksonville|  125|
+-------------+-----+



### Observation

The records were grouped by city, and only cities having more than 100 orders were displayed.

## Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column? 

In [48]:
order_items_df.agg(
    min("unit_price").alias("Minimum Price"),
    max("unit_price").alias("Maximum Price"),
    avg("unit_price").alias("Average Price")
).show()

+-------------+-------------+-------------+
|Minimum Price|Maximum Price|Average Price|
+-------------+-------------+-------------+
|          309|        59985|   30014.7524|
+-------------+-------------+-------------+



### Observation

The agg() function was used to calculate multiple statistics in a single operation, making aggregation more efficient.

# 11. Shuffle Concept (Q11)

## Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation? 

Shuffle is the process of redistributing data across partitions during operations like groupBy(), join(), and orderBy(). It is called a wide transformation because data moves between different partitions, which increases execution time compared to narrow transformations.

# 12. Complete Processing Pipeline (Q15)

## Q15: Write a final processing pipeline that: 
    Filters out duplicates. 
    Fills null prices with 0. 
    Groups by store_id to calculate total revenue.  [Note: I have used product_id]






In [51]:
pipeline_df = (
    order_items_df
    .dropDuplicates()
    .fillna({"unit_price": 0})
    .withColumn(
        "revenue",
        col("quantity") * col("unit_price")
    )
    .groupBy("product_id")
    .agg(
        sum("revenue").alias("Total Revenue")
    )
)

pipeline_df.show()

+----------+-------------+
|product_id|Total Revenue|
+----------+-------------+
| PROD00086|      5229010|
| PROD00174|      5608216|
| PROD00045|      5131041|
| PROD00157|      5520143|
| PROD00019|      4100761|
| PROD00063|      5001316|
| PROD00161|      5176469|
| PROD00235|      5021540|
| PROD00153|      5512671|
| PROD00240|      4679553|
| PROD00024|      4511971|
| PROD00020|      4103606|
| PROD00139|      5710407|
| PROD00022|      5282572|
| PROD00028|      4232638|
| PROD00209|      4535084|
| PROD00239|      7114338|
| PROD00133|      3100095|
| PROD00154|      4867925|
| PROD00101|      5020053|
+----------+-------------+
only showing top 20 rows


### Observation

A complete Spark pipeline was created by combining duplicate removal, null handling, derived column creation, and aggregation into a single workflow.

# 13. Observations


# Observations

- Successfully loaded multiple datasets into Spark DataFrames.
- Performed duplicate removal and null value handling.
- Applied filtering and aggregation operations.
- Joined multiple datasets for analysis.
- Implemented a complete Spark processing pipeline.
- Learned how Spark transformations support data engineering workflows.

# 14. Learning Outcomes


# Learning Outcomes

Through this assignment, I learned how to:

- Work with Spark DataFrames.
- Perform data cleaning operations.
- Apply filtering and aggregation techniques.
- Handle timestamp conversion.
- Understand Spark transformations and shuffle operations.
- Build a simple end-to-end data processing pipeline.

# 15. Conclusion

# Conclusion

This assignment provided practical experience in using PySpark for data cleaning, transformation, and aggregation. It also helped me understand how Spark DataFrames can be used to process data efficiently and build simple data engineering workflows.